In [1]:
import os; os.environ['WORKDIR'] = "/home/choij/workspace/ChargedHiggsAnalysis" 
import sys; sys.path.insert(0, os.environ['WORKDIR'])

import torch
from torch_geometric.data import Data

from ROOT import TFile
from libPython.DataFormat import Particle
from libPython.DataFormat import get_muons, get_electrons, get_jets
from libPython.Selection import select
from libPython.Preprocessor import evt_to_graph
from libPython.MLTools import ParticleNet
from libPython.MLTools import predict_proba
from libPython.HistTools import HistogramWriter

Welcome to JupyROOT 6.26/06


In [2]:
MASSPOINT = "MHc-130_MA-90"
BACKGROUND = "TTLL_powheg"
LAYERS = 128
OPTIMIZER = "Adam"
INITIAL_LR = "0.001"
SCHEDULER = "StepLR"
NBATCH = 1024

In [3]:
f_sig = TFile.Open(f"{os.environ['WORKDIR']}/SelectorOutput/2017/Skim3Mu__/Selector_TTToHcToWAToMuMu_{MASSPOINT}.root")
f_bkg = TFile.Open(f"{os.environ['WORKDIR']}/SelectorOutput/2017/Skim3Mu__/Selector_{BACKGROUND}.root")

model_path = f"{os.environ['WORKDIR']}/.models/All/{MASSPOINT}_vs_{BACKGROUND}/ParticleNet_nhidden-{LAYERS}_{OPTIMIZER}_initial_lr-{str(INITIAL_LR).replace('.', 'p')}_{SCHEDULER}_nbatch-{NBATCH}.pt"
model = ParticleNet(num_features=9, num_classes=2, hidden_channels=LAYERS)
model.load_state_dict(torch.load(model_path, map_location=torch.device('cpu')))
model.eval()
writer = HistogramWriter(outfile=f"{os.environ['WORKDIR']}/triLepRegion/test.root")

In [4]:
def getScore(objects, isSignal):
    node_list = []
    for object in objects:
        node_list.append([object.Pt(),
                          object.Eta(),
                          object.Phi(),
                          object.M(),
                          object.Charge(),
                          object.IsMuon(),
                          object.IsElectron(),
                          object.IsJet(),
                          object.BtagScore()])
    data = evt_to_graph(node_list, y=int(isSignal), k=3)
    score = predict_proba(model, data.x, data.edge_index)
    return score

In [5]:
def Loop(evt, writer, writerPrefix, isSignal):
    muons = get_muons(evt)
    electrons = get_electrons(evt)
    jets, bjets = get_jets(evt)
    METv = Particle(evt.METvPt, 0., evt.METvPhi, 0.)
    
    region = select("3Mu", evt, muons, electrons, jets, bjets, "tight")
    if not region:
        return None
    
    score = getScore(muons+electrons+jets, isSignal)
    
    writer.fill_muons(f"{writerPrefix}/{region}/muons", muons, evt.GenWeight*evt.TrigLumi)
    writer.fill_electrons(f"{writerPrefix}/{region}/electrons", electrons, evt.GenWeight*evt.TrigLumi)
    writer.fill_jets(f"{writerPrefix}/{region}/jets", jets, evt.GenWeight*evt.TrigLumi)
    writer.fill_jets(f"{writerPrefix}/{region}/bjets", bjets, evt.GenWeight*evt.TrigLumi)
    writer.fill_object(f"{writerPrefix}/{region}/METv", METv, evt.GenWeight*evt.TrigLumi)
    writer.fill_hist(f"{writerPrefix}/{region}/score", score, evt.GenWeight*evt.TrigLumi, 100, 0., 1.)

In [6]:
for evt in f_sig.Events:
    Loop(evt, writer, "signal", isSignal=True)
for evt in f_bkg.Events:
    Loop(evt, writer, "background", isSignal=False)
f_sig.Close()
f_bkg.Close()
writer.close()

Saving histograms in /home/choij/workspace/ChargedHiggsAnalysis/triLepRegion/test.root...
